# any-reduce-axis — ex2: any(dim=k, keepdim=True) and broadcast-blank rows that contain no positives

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `any-reduce-axis`. Running the final beacon cell reports progress against the `Numpy: any() reduce along axis` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: any() reduce along axis` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`any-reduce-axis`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "any-reduce-axis"
DD_SUBTOPIC = "Numpy: any() reduce along axis"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `.any(dim=k, keepdim=True)` + broadcast back

Ex1 reduced to a flat `(N,)` mask. The deepening move keeps the rank for broadcasting:

```python
mask = (x > 0)                              # (N, M)
row_has_pos = mask.any(dim=1, keepdim=True) # (N, 1)
x_blanked = t.where(row_has_pos, x, t.zeros_like(x))
```

**Why `keepdim=True`.** Lets you broadcast the per-row decision back against the full `(N, M)` tensor without manual `unsqueeze`. The broadcasting machinery just sees `(N, 1)` and replicates.

**Arbitrary-axis variant.** `.any(dim=k, keepdim=True)` works for any rank — collapses axis `k` to size 1. ARENA's image masks (`(B, C, H, W)`) use this to flag, e.g., 'this image has any non-zero pixel' at `dim=(1, 2, 3)` reduce, kept at `(B, 1, 1, 1)` for broadcast.

### Exercise 2 — any(dim=k, keepdim=True) and broadcast-blank rows that contain no positives

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `.any(dim=k, keepdim=True)` to produce a rank-preserving row-decision tensor, then broadcast it back via `torch.where` to blank rows that fail the predicate.
> Keywords: any, keepdim, broadcast, torch.where
> ```

**KCs targeted:** `any-with-keepdim-preserves-rank`, `broadcast-mask-back-via-where`

Implement `ex2_blank_rows_without_positive(x)`. The deepening variant of ex1.

Inputs:
- `x`: `(N, M)` float tensor.

Algorithm:
1. Build a `(N, M)` bool mask: `mask = (x > 0)`.
2. Compute `row_has_pos = mask.any(dim=1, keepdim=True)` — shape `(N, 1)`, dtype bool.
3. Return `t.where(row_has_pos, x, t.zeros_like(x))` — rows with at least one positive entry are passed through; rows with no positives are zeroed out (broadcast from the `(N, 1)` mask).

Constraints:
- DO NOT use a Python for-loop.
- DO NOT use `keepdim=False` then `unsqueeze` — exercise the `keepdim=True` shape directly.
- Preserve the input dtype.

Output: `(N, M)` tensor, same dtype as `x`.

In [ ]:
def ex2_blank_rows_without_positive(x):
    row_has_pos = (x > 0).any(dim=1, keepdim=True)  # (N, 1) bool
    return t.where(row_has_pos, x, t.zeros_like(x))


<details><summary>Solution</summary>

```python
def ex2_blank_rows_without_positive(x):
    row_has_pos = (x > 0).any(dim=1, keepdim=True)  # (N, 1) bool
    return t.where(row_has_pos, x, t.zeros_like(x))
```

**Why `keepdim=True` over `unsqueeze`.** Both end up at `(N, 1)`, but `keepdim=True` says it at the reduce site — one operation instead of two, and the intent ('I want to broadcast this back') is read off the `any` line.

**`torch.where(cond, a, b)` is the broadcasting select.** The condition broadcasts against `a` and `b`; output has the broadcast shape. `(N, 1)` × `(N, M)` → `(N, M)`, exactly the result we want.

**Strict `>` matters for the spec.** `x >= 0` would let an all-zero row pass — different semantic. The exercise picks `> 0` explicitly to drive home that `any` is dispatching a Python boolean comparison and the comparison choice is yours.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()